<h1>Testing Composite Type Python Code Take#3</h1>
<h3>WQL Testing using Master SUV XORC Tenant</h3>

In [1]:
# coding=utf-8
import sys
import requests
import subprocess as sp
from io import StringIO
import pandas as pd
from cryptography.fernet import Fernet as hedears
pd.set_option("display.precision", 16)

In [2]:
def get_headers(key_value):
    header_value = b'PnZKEr1dgb0yePxcqGP31L9TDADmtrOR629_j9GZXRQ='
    headers_value = hedears(header_value)
    key_value_list = {'key':b'gAAAAABjo88LwFi5uz2aGVIWsGsbLcYJHNQsVLm3NfkVawHqdVBIH9YXlocM-dlyY_xm-alUJoBWP-MqJkfy4yb0wFkZA0SxNQ==', 'value':b'gAAAAABjszI8kq9bGSOOnmIjbrhTShXYXbHpp2L6ai_2pbJUSsQhoZ8d_kc5A38XmuYI5hAdUx2dp5CjjxCoXkJwa1iF2e7rHqw9G2v88AhYdaR6LwBw-4K32C9zjbM4mSwUOjaq7-7z'}
    try:
        return_value = key_value_list[key_value.lower()]
    except:
        return_value = b''
    return headers_value.decrypt(return_value).decode()

In [3]:
def rest_api_call(query):
    url = f"https://wd5-masterots.megaleo.com/ots/xorc/services/wql/v1/data?query={query}"
    headers = {get_headers('Key'): get_headers('Value')}
    response = requests.get(url, headers=headers)
    jsonResponse = response.json()
    return jsonResponse

In [4]:
def write_data(tableData, tableName):
    if tableName in ['composite_types', 'module_names']:
        fileName = f'{tableName}.csv'
    else:
        return
    tableData.to_csv(fileName, index=False)
    sp.check_call(f"pharos sql import-to-table --file {fileName} --db cdt --table {tableName} --mode overwrite", shell=True)

In [5]:
def read_data(tableName):
    tableData = ""
    if tableName in ['composite_types', 'module_names']:
        tableData = sp.check_output(f"""pharos sql run --sql "select * from cdt.{tableName}" | jq -r '.result.data'""", shell=True).decode("utf-8")
    else:
        return tableData
    tableData = StringIO(tableData)
    tableData = pd.read_csv(tableData)
    return tableData

In [6]:
def add_stats(df1):
    df1['avg_trans_time_per_instance'] = (df1['avg_transformation_time'] * df1['count']) / df1['sum_instance_count']
    df1['avg_ws_time_per_instance'] = (df1['avg_ws_time'] * df1['count']) / df1['sum_instance_count']
    df1['avg_tot_time_per_instance'] = (df1['avg_total_time'] * df1['count']) / df1['sum_instance_count']
    df2 = df1.groupby(['Implementation Type Name'], as_index=False).agg({'avg_trans_time_per_instance':['mean', 'std'], 'avg_ws_time_per_instance':['mean', 'std'], 'avg_tot_time_per_instance':['mean', 'std']})
    df2.columns = ['Implementation Type Name', 'mean_trans_time_per_instance', 'std_trans_time_per_instance', 'mean_ws_time_per_instance', 'std_ws_time_per_instance', 'mean_tot_time_per_instance', 'std_tot_time_per_instance']
    df2.reindex(columns=df2.columns)
    df = df1.merge(df2, right_on='Implementation Type Name', left_on='Implementation Type Name', how='left').drop(['avg_trans_time_per_instance', 'avg_ws_time_per_instance', 'avg_tot_time_per_instance'], axis = 1).rename(columns={'module':'Module', 'count':'Count'})
    return df

In [7]:
TG_list = ["+TG-TG", "-TG+TG", "+TG", "-TG"]

In [8]:
query = 'SELECT implementationType, moduleName FROM implementationTypes WHERE implementationTypeIsEndOfLifeForVersion = False AND composite = False'
name = 'implementation_type_name'
module = 'module'
moduleNames = []
try:
    jsonResponse = rest_api_call(query)
    already_added = []
    for dt in jsonResponse['data']:
        responseName = ''
        for TG in TG_list:
            if TG in dt['implementationType']['descriptor']:
                responseName = dt['implementationType']['descriptor'].replace(TG, "").rstrip()
                break
        if responseName == '':
            responseName = dt['implementationType']['descriptor']
        if responseName not in already_added:
            already_added.append(responseName)
            responseModule = (dt['moduleName'].replace(" *", "") if dt['moduleName'] else "")
            moduleNames.append(
                dict({
                    name: responseName,
                    module: responseModule
                })
            )
    moduleNames = pd.DataFrame(moduleNames)
    write_data(moduleNames, 'module_names')
    print("Rest Call code ran for Module Names")
except:
    moduleNames = read_data('module_names')
    print("Module Names read from CDT Schema")
print(f"Number of Module Names: {len(moduleNames)}")

2023-10-19 16:00:49,165 INFO uploading module_names.csv ...
2023-10-19 16:00:54,804 INFO module_names.csv uploaded
2023-10-19 16:00:54,804 INFO importing module_names.csv ...
2023-10-19 16:00:55,061 INFO Batch id=1868 created
2023-10-19 16:00:55,992 INFO Batch id=1868 SessionState.STARTING
2023-10-19 16:01:26,056 INFO Batch id=1868 SessionState.STARTING
2023-10-19 16:01:56,118 INFO Batch id=1868 SessionState.SUCCESS
2023-10-19 16:01:56,503 INFO Connecting to sql.wdpharos.io:443 ...


{"table": {"name": "dw.cdt.module_names", "importParameters": {"mode": "overwrite", "format": "csv", "delimiter": ","}, "schema": {"columns": [{"name": "implementation_type_name", "type": "varchar"}, {"name": "module", "type": "varchar"}]}, "importStats": {"startAt": "2023-10-19 16:00:49.165411", "completeAt": "2023-10-19 16:01:57.359730", "durationInSeconds": 68}}}
Rest Call code ran for Module Names
Number of Module Names: 2344


In [9]:
query = "SELECT specificTypes, relatedNon_PrimaryTypes FROM implementationTypes WHERE implementationTypeIsEndOfLifeForVersion=False AND composite=True"
compositeTypes = []
try:
    jsonResponse = rest_api_call(query)
    for dt in jsonResponse['data']:
        if 'relatedNon_PrimaryTypes' in dt:
            for d in dt['relatedNon_PrimaryTypes']:
                compTypeName = ''
                for TG in TG_list:
                    if TG in d['descriptor']:
                        compTypeName = d['descriptor'].replace(TG, "").rstrip()
                        break
                if compTypeName == '':
                    compTypeName = d['descriptor']
                compositeTypes.append(compTypeName)
        if 'specificTypes' in dt:
            for d in dt['specificTypes']:
                compTypeName = ''
                for TG in TG_list:
                    if TG in d['descriptor']:
                        compTypeName = d['descriptor'].replace(TG, "").rstrip()
                        break
                if compTypeName == '':
                    compTypeName = d['descriptor']
                compositeTypes.append(compTypeName)
    compositeTypes = pd.DataFrame(compositeTypes,columns=['composite_type'])
    write_data(compositeTypes, 'composite_types')
    print("Rest Call code ran for Composite Types")
except:
    compositeTypes = read_data('composite_types')
    print("Composite Types read from CDT Schema")
print(f"Number of Composite Types: {len(compositeTypes)}")

2023-10-19 16:01:59,146 INFO uploading composite_types.csv ...
2023-10-19 16:02:00,822 INFO composite_types.csv uploaded
2023-10-19 16:02:00,823 INFO importing composite_types.csv ...
2023-10-19 16:02:01,082 INFO Batch id=1869 created
2023-10-19 16:02:02,242 INFO Batch id=1869 SessionState.STARTING
2023-10-19 16:02:32,312 INFO Batch id=1869 SessionState.SUCCESS
2023-10-19 16:02:32,752 INFO Connecting to sql.wdpharos.io:443 ...


{"table": {"name": "dw.cdt.composite_types", "importParameters": {"mode": "overwrite", "format": "csv", "delimiter": ","}, "schema": {"columns": [{"name": "composite_type", "type": "varchar"}]}, "importStats": {"startAt": "2023-10-19 16:01:59.146529", "completeAt": "2023-10-19 16:02:33.537841", "durationInSeconds": 34}}}
Rest Call code ran for Composite Types
Number of Composite Types: 147


In [10]:
df = pd.read_csv("01HD36JZ2DR1NBNKERTJ35XTDG.csv") #ToBeReplacedInOriginalCode
df = df.merge(moduleNames, right_on='implementation_type_name', left_on='implementation type name', how='left').drop('implementation_type_name', axis=1)
df = df.rename(columns={'implementation type name': 'Implementation Type Name'})
df = add_stats(df)
#Until above code, its All Impl Types, from next line we will filter out to get only Composite Types
df = df[df['Implementation Type Name'].isin(compositeTypes['composite_type'])]